# GraphMS-Net — Complete Evaluator Colab

This notebook runs the **frozen GraphMS v3.5.1 Hybrid research pipeline** from a clean GitHub clone.

It is designed for a guide/evaluator who has:
- a CUDA-enabled Google Colab runtime,
- one co-registered **FLAIR + T1 + T2** NIfTI triplet,
- no access to the original GraphMS research Drive.

The notebook verifies the repository, replays the frozen Stage16 aggregate evidence, downloads the required frozen neural fold from the public GitHub Release with integrity checks, executes the complete patient path, and displays the generated outputs.

**Scientific scope:** the reported segmentation performance remains the frozen **development five-fold CV** result. This notebook is a reproducibility/evaluator runner; it does not establish external validation or clinical suitability.

## 0. Before running

In Colab choose **Runtime → Change runtime type → GPU**.

For the accepted reproducibility demonstration, use:
- case ID: `MSLesSeg_P10_T1`
- `*_0000.nii.gz` = FLAIR
- `*_0001.nii.gz` = T1
- `*_0002.nii.gz` = T2

For a case outside the frozen 93-case development registry, an explicit fold `0..4` is required because no new-patient ensemble rule was validated.

In [ ]:
import subprocess, sys

print("Checking NVIDIA GPU...")
subprocess.run(["nvidia-smi"], check=True)

## 1. Clone the frozen repository

This always starts from the public `main` branch so the notebook does not depend on a local copy.

In [ ]:
from pathlib import Path
import os, shutil, subprocess

REPO = Path("/content/GraphMS-Net")
if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(
    ["git", "clone", "--branch", "main", "--single-branch",
     "https://github.com/sath17-o/GraphMS-Net.git", str(REPO)],
    check=True,
)

os.chdir(REPO)
print("Repository:", REPO)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

## 2. Install the accepted inference environment

The accepted CUDA environment uses **PyTorch 2.8.0 / CUDA 12.6** and `nnunetv2==2.8.1`.

The notebook installs the CUDA PyTorch wheel first, then the repository's frozen inference and verification dependencies.

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch==2.8.0", "torchvision==0.23.0",
    "--index-url", "https://download.pytorch.org/whl/cu126"
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements-inference.txt"
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements-verify.txt"
], check=True)

print("Environment installation complete.")

## 3. Verify the CUDA neural runtime

The full 3-D GraphMS patient path is intentionally CUDA-only. If this cell fails, stop and fix the Colab GPU runtime before continuing.

In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, "scripts/check_neural_runtime.py"],
    check=True,
)

## 4. Verify the frozen research package

These checks do **not** retrain or re-select the model.

They verify the frozen manifests/audits and replay the committed Stage16 five-fold aggregation from the 93-case development evaluation table.

In [ ]:
import subprocess, sys

print("\n=== Frozen package verification ===")
subprocess.run(
    [sys.executable, "scripts/run_pipeline.py", "--mode", "verify"],
    check=True,
)

print("\n=== Stage16 frozen evaluation replay ===")
subprocess.run(
    [sys.executable, "scripts/run_pipeline.py", "--mode", "evaluation-replay"],
    check=True,
)

## 5. Patient configuration

The default below is the already accepted reproducibility case.

- For `MSLesSeg_P10_T1`, leave `FOLD = None`; the repository resolves its held-out fold automatically.
- For a case outside the frozen development registry, set `CASE_ID` and set `FOLD` explicitly to an integer from 0 to 4.

An explicit fold for an unseen case is a **selected-fold research run**, not a validated new-patient ensemble.

In [ ]:
CASE_ID = "MSLesSeg_P10_T1"
FOLD = None   # Known development case: automatic held-out-fold resolution.

print("CASE_ID:", CASE_ID)
print("FOLD:", FOLD)

## 6. Upload FLAIR, T1 and T2

Select the three co-registered NIfTI files.

For the accepted demo, upload:
- `MSLesSeg_P10_T1_0000.nii.gz` — FLAIR
- `MSLesSeg_P10_T1_0001.nii.gz` — T1
- `MSLesSeg_P10_T1_0002.nii.gz` — T2

The notebook never asks for or uses a ground-truth lesion mask.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

uploaded = files.upload()

INPUT_DIR = Path("/content/patient_input")
if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)
INPUT_DIR.mkdir(parents=True)

for name in uploaded:
    shutil.move(name, INPUT_DIR / name)

print("\nUploaded files:")
for p in sorted(INPUT_DIR.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size:,} bytes)")

## 7. Resolve the three modalities and the frozen fold

The accepted MSLesSeg/nnU-Net convention is:
- `_0000` → FLAIR
- `_0001` → T1
- `_0002` → T2

For custom filenames, rename the files to this convention before running this cell.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(REPO))

def exactly_one(suffix):
    matches = sorted(INPUT_DIR.glob(f"*{suffix}"))
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one uploaded file ending {suffix}; found {len(matches)}: "
            + ", ".join(p.name for p in matches)
        )
    return matches[0]

FLAIR = exactly_one("_0000.nii.gz")
T1 = exactly_one("_0001.nii.gz")
T2 = exactly_one("_0002.nii.gz")

from graphms.patient import resolve_fold
SELECTED_FOLD, RUN_SCOPE = resolve_fold(CASE_ID, FOLD, REPO)

print("FLAIR:", FLAIR)
print("T1:   ", T1)
print("T2:   ", T2)
print("Resolved fold:", SELECTED_FOLD)
print("Run scope:", RUN_SCOPE)

## 8. Fetch and verify the required frozen neural assets

Only the required fold is downloaded from the public `assets-v1` GitHub Release.

Every file is checked for expected size and SHA-256 identity before it is admitted into the local asset lock.

In [ ]:
import subprocess, sys

subprocess.run(
    [
        sys.executable,
        "scripts/download_release_assets.py",
        "--folds",
        str(SELECTED_FOLD),
    ],
    check=True,
)

# Independent second verification through the normal runtime asset gate.
subprocess.run(
    [
        sys.executable,
        "scripts/setup_assets.py",
        "--folds",
        str(SELECTED_FOLD),
    ],
    check=True,
)

## 9. Run the complete patient pipeline

This invokes the same guide-facing entry point documented in the repository:

**FLAIR + T1 + T2 → frozen nnU-Net/ResEncM-250 → Stage5/6 graph features → Stage7 TrueGAT → GraphMS Hybrid → Stage11 → canonical Stage12 → frozen Stage13 → report/provenance.**

No training occurs here.

In [ ]:
import subprocess, sys, shutil
from pathlib import Path

OUTPUT_DIR = REPO / "outputs" / f"{CASE_ID}_colab_evaluator"

# Notebook convenience only: ensure this execution starts with a fresh output directory.
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

cmd = [
    sys.executable,
    "run_graphms.py",
    "--case-id", CASE_ID,
    "--flair", str(FLAIR),
    "--t1", str(T1),
    "--t2", str(T2),
    "--output", str(OUTPUT_DIR),
]
if FOLD is not None:
    cmd.extend(["--fold", str(FOLD)])

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print("\nOutput directory:", OUTPUT_DIR)

## 10. Confirm completion and inspect the generated bundle

A valid completed run must contain `COMPLETE.json` with status `COMPLETE`.

In [ ]:
import json
from pathlib import Path

print("Generated outputs:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" ", p.name)

complete = json.loads((OUTPUT_DIR / "COMPLETE.json").read_text())
assert complete.get("status") == "COMPLETE", complete

print("\n=== COMPLETE.json ===")
print(json.dumps(complete, indent=2))

## 11. Inspect lesion features and Stage13 research outputs

In [ ]:
import json
import pandas as pd
from IPython.display import display

features = pd.read_csv(OUTPUT_DIR / "features.csv")
lesions = pd.read_csv(OUTPUT_DIR / "lesions.csv")
risk = json.loads((OUTPUT_DIR / "risk.json").read_text())

print("Stage12 feature table shape:", features.shape)
display(features.head())

print("\nLesion component count:", len(lesions))
display(lesions.head(10))

print("\n=== Stage13 risk.json ===")
print(json.dumps(risk, indent=2))

## 12. Display the segmentation overlay and HTML patient report

In [ ]:
from IPython.display import Image, HTML, display

print("=== Segmentation overlay ===")
display(Image(filename=str(OUTPUT_DIR / "overlay.png")))

print("\n=== Patient report ===")
display(HTML(filename=str(OUTPUT_DIR / "patient_report.html")))

## 13. Reproducibility check for the accepted demonstration case

For `MSLesSeg_P10_T1`, the repository contains an independent CUDA acceptance record. This cell compares the newly generated lesion-mask SHA-256 with the accepted prediction SHA.

For other patients, this exact reference comparison is intentionally skipped.

In [ ]:
import hashlib, json

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

if CASE_ID == "MSLesSeg_P10_T1":
    accepted = json.loads(
        (REPO / "evidence/acceptance/MSLesSeg_P10_T1_ACCEPTANCE.json").read_text()
    )
    current_hash = sha256(OUTPUT_DIR / "lesion_mask.nii.gz")
    expected_hash = accepted["prediction_sha256"]

    print("Current lesion-mask SHA-256: ", current_hash)
    print("Accepted prediction SHA-256:", expected_hash)
    print("Exact accepted-mask hash match:", current_hash == expected_hash)

    assert current_hash == expected_hash, "Generated mask does not match the accepted frozen prediction."
else:
    print("Exact accepted-mask comparison is only defined for MSLesSeg_P10_T1.")

## 14. Inspect provenance

This is where an evaluator can verify that the patient run did not use ground truth or perform new training.

In [ ]:
import json

provenance = json.loads((OUTPUT_DIR / "provenance.json").read_text())
print(json.dumps(provenance, indent=2))

assert provenance.get("ground_truth_used") is False
assert provenance.get("new_training_performed") is False

print("\nProvenance checks PASS: ground_truth_used=False, new_training_performed=False")

## 15. Download the complete output bundle

This packages the generated NIfTI, CSV, JSON, overlay, HTML report and provenance into one ZIP for inspection or handoff.

In [ ]:
import shutil
from google.colab import files

zip_base = Path("/content") / f"{CASE_ID}_GraphMS_outputs"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_DIR)

print("Created:", zip_path)
files.download(zip_path)

---

## What this notebook demonstrates

A successful run shows that the public GraphMS-Net repository can execute the frozen research pipeline from supplied FLAIR/T1/T2 inputs, acquire its own frozen neural assets from the GitHub Release, verify their identities, generate the lesion segmentation and downstream research outputs, and preserve provenance.

**It does not convert the development five-fold CV result into external validation or a clinical-performance claim.**